# Food Price Watch 06: view results and download
Reads the finished exports on Drive. Run after notebook 05. Nothing is computed or changed here. The files hold TCB prices: keep them private and out of git.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Fresh clone of main (or fast-forward if this runtime already has it).
import os
if not os.path.isdir('/content/foodprice_watch/.git'):
    !git clone --single-branch --branch main https://github.com/mrchowdhury587/foodprice_watch.git /content/foodprice_watch
else:
    !git -C /content/foodprice_watch pull --ff-only origin main
%pip install -q -r /content/foodprice_watch/requirements-colab.txt

In [ ]:
%cd /content/foodprice_watch
%env PYTHONPATH=/content/foodprice_watch/src
%env FPW_ARTIFACT_ROOT=/content/drive/MyDrive/FoodPriceWatch
from pathlib import Path
import csv, hashlib, json, shutil
ROOT = Path('/content/drive/MyDrive/FoodPriceWatch')
for folder in ['inbox/project', 'inbox/external', 'data/processed', 'outputs', 'reports/tables',
               'reports/figures', 'reports/project/tables', 'state', 'config', 'logs', 'models']:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
# First run only: copy the starting state files into Drive. Existing files and seals are never replaced.
for name in ['state/stages.json', 'config/item_driver_map.yaml', 'DECISIONS.md', 'PROGRESS.md', 'DEVIATIONS.md',
             'reports/project/tables/P2_audit.csv', 'reports/project/tables/P2_audit_summary.csv']:
    if not (ROOT / name).exists():
        shutil.copy2(name, ROOT / name)
# Remove half-written files left by an interrupted session.
for folder in ['outputs', 'data']:
    for tmp in (ROOT / folder).rglob('*.tmp'):
        tmp.unlink()
CODE_COMMIT = !git rev-parse HEAD
CODE_COMMIT = CODE_COMMIT[0]
CHECKS_PASS = False
stage_ok = {}

In [ ]:
CHECKS_PASS = False
stage_ok.clear()
!python -m fpw.check
CHECKS_PASS = get_ipython().user_ns['_exit_code'] == 0
assert CHECKS_PASS, 'Startup check failed; fix it before running stages'

## View

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
EX = ROOT / 'outputs/export'
fc = pd.read_parquet(EX / 'forecasts.parquet'); al = pd.read_csv(EX / 'alerts.csv', parse_dates=['tdate']); dr = pd.read_csv(EX / 'alert_drivers.csv')
ITEM = sorted(fc.item_id.unique())[0]    # change to any item name from the list printed below
print(len(fc.item_id.unique()), 'items; origin', fc.origin_date.max().date(), '; items:', *sorted(fc.item_id.unique())[:6], '...')
d = fc[fc.item_id == ITEM].sort_values('horizon'); print('Item:', ITEM, '| last price', d.last_price.iloc[0]); display(d[['horizon', 'forecast_low', 'forecast', 'forecast_high']].round(1))
recent = al[al.tdate >= al.tdate.max() - pd.Timedelta(days=30)]; print(len(recent), 'alert starts in the last 30 days of the data'); display(recent.sort_values('tdate', ascending=False).head(15).round(3))
g = [c for c in ['price_history', 'calendar', 'weather', 'onion_export_measures', 'exchange_rate'] if c in dr.columns]
ax = dr[g].abs().mean().plot.bar(title='Average size of each driver group on alert days'); plt.tight_layout(); plt.show()


## Interactive app (Gradio)
Starts the dashboard in this notebook and prints a link. **The link is public** while the runtime runs: anyone who has it can open it. Run the cell, wait for the link, open it. Stop the cell to close it.

In [ ]:
import gradio as gr, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
EX = ROOT / 'outputs/export'
fc = pd.read_parquet(EX / 'forecasts.parquet'); al = pd.read_csv(EX / 'alerts.csv'); dr = pd.read_csv(EX / 'alert_drivers.csv')
GROUPS = [c for c in ['price_history', 'calendar', 'weather', 'onion_export_measures', 'exchange_rate'] if c in dr.columns]
INTRO = ('# Bangladesh Food Price Watch\n'
         'Forecasts and abnormal-rise alerts for TCB Dhaka retail prices. Alerts detect a price above the expected range; they are not an early warning. '
         'The model did not beat "tomorrow = today" on the 2025-26 test.')


def forecast_view(item):
    d = fc[fc.item_id == item].sort_values('horizon')
    fig, ax = plt.subplots(figsize=(6, 3.2)); n = range(len(d))
    ax.fill_between(n, d.forecast_low, d.forecast_high, alpha=.25, label='expected range'); ax.plot(n, d.forecast, 'o-', label='forecast')
    ax.axhline(d.last_price.iloc[0], color='k', lw=1, ls='--', label='last price'); ax.set_xticks(list(n)); ax.set_xticklabels(d.horizon.astype(str))
    ax.set_xlabel('report days ahead'); ax.set_ylabel('Tk'); ax.legend(fontsize=7); plt.tight_layout()
    return d[['horizon', 'forecast_low', 'forecast', 'forecast_high']].round(1), fig


def alert_view(item, latest_only):
    a = al[al.item_id == item].sort_values('tdate', ascending=False)
    return (a.head(10) if latest_only else a).round(3)


def driver_view(item):
    d = dr[dr.item_id == item]
    fig, ax = plt.subplots(figsize=(6, 3.2))
    if len(d): d[GROUPS].abs().mean().plot.bar(ax=ax, title='Average driver size on alert days')
    else: ax.text(.5, .5, 'no alert days for this item', ha='center')
    plt.tight_layout()
    return fig, 'The part of a rise above the expected range is not explained by measured cost drivers.'


first = sorted(fc.item_id.unique())[0]
with gr.Blocks(title='Food Price Watch') as demo:
    gr.Markdown(INTRO)
    item = gr.Dropdown(sorted(fc.item_id.unique()), value=first, label='Item')
    with gr.Tab('Forecasts'):
        t, f = gr.Dataframe(), gr.Plot(); item.change(forecast_view, item, [t, f]); demo.load(forecast_view, item, [t, f])
    with gr.Tab('Alerts'):
        latest = gr.Checkbox(True, label='Latest 10 only'); a = gr.Dataframe()
        item.change(alert_view, [item, latest], a); latest.change(alert_view, [item, latest], a); demo.load(alert_view, [item, latest], a)
    with gr.Tab('Why an alert'):
        f2, note = gr.Plot(), gr.Markdown(); item.change(driver_view, item, [f2, note]); demo.load(driver_view, item, [f2, note])
demo.launch(share=True)    # prints a public gradio.live link (about 72 hours, only while this runtime is running)


## Download to this computer
Sends two zip files to your browser's Downloads folder: the three export files (for the local dashboard) and the report tables, figures and results register. The browser may ask to allow multiple downloads.

In [ ]:
from google.colab import files
import shutil
for name, src in [('export_files', ROOT / 'outputs/export'), ('report_tables_and_figures', ROOT / 'reports/project')]:
    z = shutil.make_archive('/content/' + name, 'zip', src); print(name, '->', z); files.download(z)
